A generic BB84 protocol simulation. Note that there is no quantum channel so we are faking it with Quantum circuit
Strict key generation as keys must match so even if error rates is 0 approx it is not good enough
We can run error correction when error rate is low but keeping it strict match in this impl

In [36]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import StatevectorSimulator, QasmSimulator, Aer

class BB84:
    # as we can not transfer qubits via a quantum channel in code with qiskit
    # we will use circuit object as proxy and assume EavesDropper can not see but only measure it
    def __init__(self, num_bits_of_key_desired, eavesdropper_present = True):
        self.num_bits_of_key_desired = num_bits_of_key_desired
        self.num_bits_key_at_start = 4 * self.num_bits_of_key_desired
        self.random_classical_bits = self.generate_random_classical_bits()
        self.source = Source(self.random_classical_bits)
        self.quantum_channel = self.source.encode()

        if(eavesdropper_present):
            self.eavesdropper = EavesDropper(self.quantum_channel.copy()) # only circuit instruction copy not violating no cloning
            self.quantum_channel = self.eavesdropper.intercept_resend()
        # next step create destination and send this intercepted cricuit
        self.destination = Destination(self.quantum_channel.copy())

    def generate_random_classical_bits(self):
        return np.random.randint(2, size=self.num_bits_key_at_start)
        

    def generate_key(self):
        # keygeneration upto an error rate, if accuracy needed run until keys match.
        # 1. match the source and destination bases and discard bits where they do not match
        # update their coress to math bases keep only the bits that match and discard the rest
        success = False
        source_key = self.source.update_sifted_key(self.destination.random_bases)
        destination_key = self.destination.update_sifted_key(self.source.random_bases)

        if (len(source_key) < 2*self.num_bits_of_key_desired):
            print("Not enough bits to generate key, try again, source key: ", source_key, "destination key: ", destination_key)
            source_key = None
        else:
            n = len(source_key)
            check_indices = np.arange(0, n, 2) # means start from 0 go till n but hop in steps of 2, keep indices
            key_indices   = np.arange(1, n, 2) # keeps odd numbers, keep indices
            check_source = source_key[check_indices]
            check_dest   = destination_key[check_indices]
            error_rate = np.mean(check_source != check_dest)
            source_key = source_key[key_indices]
            destination_key = destination_key[key_indices]

            if np.all(source_key == destination_key):
                print("Keys match, secure key generated successfully! ", "source key: ", source_key, "destination key: ", destination_key)
                success = True
            else:
                print("Key not secure, try again, error rate: ", error_rate, "source key: ", source_key, "destination key: ", destination_key)

        return source_key, destination_key, success


class Source:
    def __init__(self, random_classical_bits):
        self.random_classical_bits = random_classical_bits
        self.num_bits = len(random_classical_bits)
        self.random_bases = self.generate_random_bases()
        self.qubits_register = QuantumRegister(self.num_bits, "qr_source")
        self.circuit = QuantumCircuit(self.qubits_register)
        self.S_simulator = StatevectorSimulator()
        self.M_simulator = QasmSimulator()
        self.matched_bases = None
        self.sifted_classical_bits = None
        self.matched_indices = None

    def generate_random_bases(self):
        return np.random.randint(2, size=self.num_bits)

    def encode(self):
        # default state of qubit is |0> so first encode the classical bits correctly
        for i, bit in enumerate(self.random_classical_bits):
            if bit == 1:
                self.circuit.x(self.qubits_register[i])

        # now apply the bases
        for i, basis in enumerate(self.random_bases):
            if basis == 1:
                self.circuit.h(self.qubits_register[i])

        return self.circuit
    
    def update_sifted_key(self, destination_bases):
        # here use param and match source bases
        # then update the measure bits such that only matched bases bits are kept and rest are discarded
        self.matched_indices = np.where(self.random_bases == destination_bases)[0]
        self.matched_bases = self.random_bases[self.matched_indices]
        self.sifted_classical_bits = self.random_classical_bits[self.matched_indices]

        return self.sifted_classical_bits

class EavesDropper:
    def __init__(self, qubits_channel):
        self.qubits_channel = qubits_channel
        self.num_qubits = qubits_channel.num_qubits
        self.random_bases = self.generate_random_bases()
        self.classical_register = ClassicalRegister(self.num_qubits, "cr_eavesdropper")
        self.qubits_channel.add_register(self.classical_register)
        self.resend_qubits_register = QuantumRegister(self.num_qubits, "qr_eavesdropper")
        self.resend_circuit = QuantumCircuit(self.resend_qubits_register)

        self.S_simulator = StatevectorSimulator()
        self.M_simulator = QasmSimulator()

    def generate_random_bases(self):
        return np.random.randint(2, size=self.num_qubits)
    

    def intercept_resend(self):
        # measure qubits in random bases by intercepting source
        for i, basis in enumerate(self.random_bases):
            if basis == 1:
                self.qubits_channel.h(i)
            self.qubits_channel.measure(i, self.classical_register[i])

        result = self.M_simulator.run(self.qubits_channel, shots = 1).result()
        counts = result.get_counts()
        bitstring = list(counts.keys())[0]
        bitstring = bitstring[::-1]
        eavesdropper_measured_bits = np.array(list(map(int, bitstring)))

        # recreate new circuit (our proxy for channel) and resend to Destination
        # encode eavesdropper bits first
        for i, bit in enumerate(eavesdropper_measured_bits):
            if bit == 1:
                self.resend_circuit.x(self.resend_qubits_register[i])

        # now apply the eavesdropper bases
        for i, basis in enumerate(self.random_bases):
            if basis == 1:
                self.resend_circuit.h(self.resend_qubits_register[i])

        return self.resend_circuit
    

class Destination:
    def __init__(self, qubits_channel):
        self.qubits_channel = qubits_channel
        self.num_qubits = qubits_channel.num_qubits
        self.random_bases = self.generate_random_bases()
        self.classical_register = ClassicalRegister(self.num_qubits, "cr_destination")
        self.qubits_channel.add_register(self.classical_register)

        self.S_simulator = StatevectorSimulator()
        self.M_simulator = QasmSimulator()
        self.measured_bits = self.measure()
        self.matched_bases = None
        self.sifted_measured_bits = None
        self.matched_indices = None

    def generate_random_bases(self):
        return np.random.randint(2, size=self.num_qubits)
    
    def measure(self):
        for i, basis in enumerate(self.random_bases):
            if basis == 1:
                self.qubits_channel.h(i)
            self.qubits_channel.measure(i, self.classical_register[i])

        result = self.M_simulator.run(self.qubits_channel, shots = 1).result()
        counts = result.get_counts()
        bitstring = list(counts.keys())[0]
        bitstring = bitstring[::-1]
        destination_measured_bits = np.array(list(map(int, bitstring)))

        return destination_measured_bits

    def update_sifted_key(self, source_bases):
        # here use param and match destination bases
        # then update the measure bits such that only matched bases bits are kept and rest are discarded
        self.matched_indices = np.where(self.random_bases == source_bases)[0]
        self.matched_bases = self.random_bases[self.matched_indices]
        self.sifted_measured_bits = self.measured_bits[self.matched_indices]
        return self.sifted_measured_bits

iteration = 0
while True:
    bb84protocol = BB84(num_bits_of_key_desired=10, eavesdropper_present=True)
    iteration += 1
    print("iteration: ", iteration)
    source_key, destination_key, success = result = bb84protocol.generate_key()
    if success:
        break


iteration:  1
Not enough bits to generate key, try again, source key:  [0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 1] destination key:  [1 0 1 0 0 1 0 0 0 0 1 0 0 1 1 1]
iteration:  2
Key not secure, try again, error rate:  0.18181818181818182 source key:  [0 0 0 1 0 0 1 0 0 1 1] destination key:  [0 0 1 1 0 0 1 0 1 0 0]
iteration:  3
Key not secure, try again, error rate:  0.25 source key:  [1 0 1 0 1 1 0 1 1 0 1] destination key:  [1 1 1 1 1 1 0 0 0 0 1]
iteration:  4
Key not secure, try again, error rate:  0.2 source key:  [1 0 1 1 0 1 0 1 1 0] destination key:  [1 1 1 1 0 1 0 0 1 0]
iteration:  5
Key not secure, try again, error rate:  0.45454545454545453 source key:  [0 0 1 0 0 0 0 0 0 1] destination key:  [0 0 1 0 0 1 0 0 0 1]
iteration:  6
Not enough bits to generate key, try again, source key:  [0 1 0 1 0 0 0 0 0 1 0 1 1 1 1 1 1 0 1] destination key:  [0 1 0 1 1 0 0 0 1 1 0 1 0 0 0 1 0 0 1]
iteration:  7
Key not secure, try again, error rate:  0.3 source key:  [0 1 0 1 1 0 0 1 0 0] destinat